# 🎙️ Studio Editoriale AI

Questo notebook esegue l'applicazione **Studio Editoriale AI** su Google Colab con accelerazione GPU, scaricando automaticamente il codice e le voci dal repository GitHub.

### ⚙️ Prima di iniziare:
1. Assicurati che il runtime sia impostato su **GPU T4** (*Runtime* -> *Cambia tipo di runtime* -> Seleziona **T4 GPU** -> *Salva*).

### 1. Clonazione del Repository da GitHub
Esegui questa cella per scaricare automaticamente tutti i file del progetto in un click.

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/emanuelec807/AIStudioEbook.git"
REPO_NAME = "AIStudioEbook"

if not os.path.exists(f"/content/{REPO_NAME}"):
    print(f"⏳ Clonazione del repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} /content/{REPO_NAME}
else:
    print(f"🔄 Aggiornamento repository {REPO_NAME}...")
    %cd /content/{REPO_NAME}
    !git pull

%cd /content/{REPO_NAME}
print(f"\n✅ Repository pronto nella cartella: {os.getcwd()}")

### 2. Installazione delle Dipendenze
Esegui questa cella per installare tutte le librerie necessarie (circa 1 minuto). Al termine effettuerà la verifica automatica.

In [ ]:
# 1. Installa Coqui TTS, Kokoro e versioni stabili di numpy/scipy
print("⏳ Installazione dipendenze in corso...")
!pip install -q "numpy<2.0.0" "scipy<1.15.0" coqui-tts kokoro soundfile transformers flask flask-cors pydub EbookLib beautifulsoup4 accelerate requests
!apt-get install -y -qq ffmpeg espeak-ng

# 2. Ricarica la cache dei moduli di Python
import sys
import site
import importlib
importlib.invalidate_caches()

# 3. Test di verifica import
try:
    import torch
    import soundfile
    import TTS
    from TTS.api import TTS as CoquiTTS
    import kokoro

    print("\n" + "="*50)
    print("🎉 TUTTE LE LIBRERIE INSTALLATE E VERIFICATE!")
    print(f"✅ PyTorch (GPU CUDA: {torch.cuda.is_available()})")
    print(f"✅ Coqui XTTS: {TTS.__version__}")
    print(f"✅ Kokoro: {kokoro.__version__}")
    print("="*50)
except (ImportError, ModuleNotFoundError):
    print("\n🔄 Riavvio rapido del runtime in corso per applicare le librerie...")
    import os
    os.kill(os.getpid(), 9)

### 3. (Opzionale) Installazione e Avvio di Ollama per TranslateGemma (12B & 4B)
Se desideri utilizzare la traduzione AI con **TranslateGemma 12B e 4B** su Colab, esegui questa cella.

In [ ]:
import subprocess
import time
import requests
import os

# 1. Installa zstd ed Ollama su Linux
print("⏳ Installazione zstd ed Ollama...")
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Avvia il server Ollama in background
print("⏳ Avvio server Ollama...")
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=open("ollama.log", "w"))

# Attendi che il server Ollama risponda
print("⏳ Inizializzazione Ollama in corso...")
for _ in range(15):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)

# 3. Scarica TranslateGemma 12B e 4B
print("⏳ Download TranslateGemma 12B...")
!ollama pull translategemma:12b
print("⏳ Download TranslateGemma 4B...")
!ollama pull translategemma:4b

print("\n" + "="*50)
print("🎉 OLLAMA & TRANSLATEGEMMA (12B + 4B) PRONTI E ATTIVI!")
print("="*50)

### 4. Recupero IP del Tunnel
Localtunnel richiede di inserire l'IP pubblico del server Colab per sbloccare l'accesso al primo clic. Esegui la cella qui sotto e **copia l'IP stampato**.

In [ ]:
# Copia questo indirizzo IP, ti servirà per accedere al link pubblico di localtunnel
!curl ipv4.icanhazip.com

### 5. Avvio del Server e Link Pubblico
Esegui questa cella per avviare il server in background ed esporre l'interfaccia web a internet. 

**Come accedere:**
1. Fai clic sul link `.localtunnel.me` generato in fondo alla cella.
2. Incolla l'IP copiato nel passaggio precedente nella schermata di sicurezza di localtunnel.
3. Clicca su **Submit** e l'applicazione si aprirà!

In [ ]:
import subprocess
import time
import os

# 1. Crea cartelle di output se non esistono
os.makedirs("audiolibri_output", exist_ok=True)
os.makedirs("audiolibriEpub", exist_ok=True)

# Scarica file voce_rif_female.wav se non presente per garantire un preset iniziale
if not os.path.exists("voce_rif_female.wav"):
    print("⏳ Download voce di riferimento preset iniziale...")
    !wget -q https://github.com/DeepMount00/Sibilia-TTS/raw/main/voce_rif_female.wav -O voce_rif_female.wav
    print("✅ Voce preset scaricata!")

# 2. Avvia il server Flask in background scrivendo i log nel file flask.log
print("⏳ Avvio del server Python...\n")
subprocess.Popen(["python", "server.py"], stdout=open("flask.log", "w"), stderr=open("flask.log", "w"))

# 3. Attendi che il server si inizializzi
time.sleep(5)
print("✅ Server Flask pronto!")

# 4. Avvia localtunnel direttamente senza richiedere conferme (flag --yes)
print("🔗 Clicca sul link qui sotto appena compare:")
!npx --yes localtunnel --port 5000